<a href="https://colab.research.google.com/github/osleysorio/itacedemy_repo1_osley/blob/main/sprint%207/ejercicio2_N2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Algoritmo  y promgramas
Ejercicio 2: Clasificación de clientes de seguros

## Importar librerias

In [1]:
import pandas as pd
import numpy as np

## Lista de de clientes

In [2]:
clients = [
    {"nom": "Ana", "edat": 42, "codi": 0},
    {"nom": "Carlos", "edat": 30, "codi": 2},
    {"nom": "Isabel", "edat": 55, "codi": 3},
    {"nom": "Jorge", "edat": 40, "codi": 1},
    {"nom": "Marta", "edat": 28, "codi": 0}

]

In [3]:
def cliente_df(dic_clientes):
  # 2. Convertir la lista de clientes a DataFrame
  df_clients = pd.DataFrame(dic_clientes)
  return df_clients

## Diccionario con las categorías del sistema

In [4]:
nivells_seguro = {
    0: {"categoria": "Baix risc", "preu_base": 120, "fraude": 0.01},
    1: {"categoria": "Risc mitjà", "preu_base": 200, "fraude": 0.05},
    2: {"categoria": "Risc alt", "preu_base": 350, "fraude": 0.15},
    3: {"categoria": "Risc crític", "preu_base": 500, "fraude": 0.30}
}

* Tomar la lista nativa de diccionarios que contiene los datos de identidad de los asegurados.
* Instanciar un  DataFrame donde las claves actúan como columnas.
* Cargar el diccionario estructurado de políticas de seguros basando sus filas en los índices numéricos correspondientes al código de riesgo (0, 1, 2, 3).

In [5]:
def nivel_seguro_df(dic_nivel_seguro):

# 3. Convertir el diccionario de seguros a DataFrame usando el índice (codi)
  df_seguros = pd.DataFrame.from_dict(nivells_seguro, orient='index')
  return df_seguros

## Clasificar cada cliente utilizando niveles_seguro ( sense if/elif).

* Identificar el punto de acoplamiento común: la columna codi en los clientes y el Índice en la tabla de seguros.
* Cruzar horizontalmente ambos conjuntos de datos combinando la información de cada persona con su respectiva fila de riesgo.
* Retornar una única matriz unificada que hereda los campos de categoría, precio base y probabilidad de fraude correspondientes.

In [6]:
def clasificacio_cliente(df_clients, df_seguros):
  # 4. Fusionar ambos DataFrames usando la columna 'codi'
  df_final = df_clients.merge(df_seguros, left_on='codi', right_index=True)
  return df_final

## Calcular el precio final de la póliza con la fórmula dada.

* Leer vectorialmente los valores de las columnas preu_base y edat de todos los elementos simultáneamente.
* Dividir cada edad entre 100 para computar la tasa incremental ponderada.
* Sumar la unidad al ratio calculado y multiplicarlo por la tarifa inicial de la póliza.
* Insertar el resultado final calculado en una nueva columna dedicada llamada preu_final.

In [7]:
def calcular_precio_final(df_final):
  # 5. Calcular el precio final de la póliza
  df_final['preu_final'] = df_final['preu_base'] * (1 + df_final['edat']/100)
  return df_final

## Ordenar a los clientes:
* Evaluar primero la columna fraude organizando las filas con los ratios más peligrosos arriba.
* En caso de empate enla comparacion del 'fraude', evaluar secundariamente la columna preu_final.
* Reordenar el DataFrame  descendente para fraude y ascendente para el importe económico.

In [8]:
def ordenar_clientes(df_final):
    # - Por probabilidad de fraude (descendente).
    # - Por precio final (ascendente por defecto).
    df_ordenado = df_final.sort_values(by=['fraude', 'preu_final'], ascending=[False, True])
    return df_ordenado

* Buscar y aislar el valor numérico máximo de la columna fraude, aislando los nombres que igualen dicho límite.
* Localizar el valor monetario máximo en preu_final para identificar al cliente con mayor impacto financiero.
* normalizar el precio final entre un rango absoluto de \(0\) a \(1\) para neutralizar las magnitudes físicas de las variables.
* Sumar la probabilidad de fraude al precio escalado creando un índice ponderado único de criticidad.
* Filtrar e imprimir en consola el perfil o perfiles que acumulen el valor máximo de decisión para auditoría directa.

In [9]:
def analisis_clientes(df_final):
    # 1.- Qué cliente tiene más probabilidad de fraude
    #  Encontrar el valor máximo de la probabilidad de fraude
    max_fraude = df_final['fraude'].max()

    #  Filtrar todas las filas que tengan exactamente ese valor máximo
    clientes_max_fraude = df_final[df_final['fraude'] == max_fraude]

    print(f"Se encontraron {len(clientes_max_fraude)} clientes con la probabilidad máxima de {max_fraude}:")
    print(clientes_max_fraude[['nom', 'fraude']])

   # 2. Cliente con el precio final más alto
    max_precio = df_final['preu_final'].max()
    clientes_max_precio = df_final[df_final['preu_final'] == max_precio]
    print(f"-> Precio final más alto ({max_precio} €):")
    # C llamamos a clientes_max_precio correctamente
    print(clientes_max_precio[['nom', 'preu_final']].to_string(index=False))
    print("-" * 50)


    #3. Que cliente seria prioritario revisar manualmente

    #  Normalizar el precio entre 0 y 1
    df_final['precio_normalizado'] = (df_final['preu_final'] - df_final['preu_final'].min()) / (df_final['preu_final'].max() - df_final['preu_final'].min())

    #  Calcular la métrica de decisión
    df_final['valor_decision'] = df_final['fraude'] + df_final['precio_normalizado']

    #  Encontrar el valor máximo de la métrica
    max_decision = df_final['valor_decision'].max()

    #  Filtrar los clientes con el valor máximo
    clientes_max_desicion = df_final[df_final['valor_decision'] == max_decision]

    #  Mostrar el valor máximo en consola
    print(f"Los clientes con maximo valor de decision: {max_decision}, para atenderlos manualmente:")

    # clientes con más 'valor_decision'
    print(clientes_max_desicion[['nom', 'valor_decision']])



##  Crear un DataFrame o diccionario con la información final.

In [10]:
def main():
    df_clients = cliente_df(clients)
    df_seguros = nivel_seguro_df(nivells_seguro)
    df_final = clasificacio_cliente(df_clients, df_seguros)
    df_final = calcular_precio_final(df_final)

    # Aplicamos la nueva función de ordenación
    df_ordenado = ordenar_clientes(df_final)

    print(df_ordenado)
    print("\nAnálisis de clientes:")
    print(analisis_clientes(df_final))



In [11]:
# Llamada al método principal
if __name__ == "__main__":
    main()

      nom  edat  codi    categoria  preu_base  fraude  preu_final
2  Isabel    55     3  Risc crític        500    0.30       775.0
1  Carlos    30     2     Risc alt        350    0.15       455.0
3   Jorge    40     1   Risc mitjà        200    0.05       280.0
4   Marta    28     0    Baix risc        120    0.01       153.6
0     Ana    42     0    Baix risc        120    0.01       170.4

Análisis de clientes:
Se encontraron 1 clientes con la probabilidad máxima de 0.3:
      nom  fraude
2  Isabel     0.3
-> Precio final más alto (775.0 €):
   nom  preu_final
Isabel       775.0
--------------------------------------------------
Los clientes con maximo valor de decision: 1.3, para atenderlos manualmente:
      nom  valor_decision
2  Isabel             1.3
None
